# Day 1-3: Transformer 모델 (BERT/KoBERT)

**강의 시간**: 1.5시간  
**학습 목표**:
- BERT의 핵심 개념 이해
- Hugging Face Transformers 라이브러리 사용
- KoBERT Fine-tuning으로 성능 향상
- 베이스라인 대비 성능 비교

**사전 요구사항**: Day 1-2 베이스라인 완료

**예상 성능**:
- Baseline (TF-IDF): F1 ~0.80
- **BERT**: F1 ~0.90 🚀


## 🔧 0. 환경 재설정



### 필수 라이브러리 설치

In [ ]:
# Transformers 라이브러리 설치 (약 1분 소요)
%pip install -q transformers datasets accelerate

print("✅ Transformers 라이브러리 설치 완료!")

In [ ]:
# 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import warnings

warnings.filterwarnings('ignore')

# 1) 폰트 파일 직접 다운로드 (런타임 재시작 불필요)
!wget -q -O NanumGothic.ttf -L "https://fonts.gstatic.com/ea/nanumgothic/v5/NanumGothic-Regular.ttf"

import matplotlib.font_manager as fm

# 폰트 파일 경로
font_path = "NanumGothic.ttf"

# 폰트 매니저에 폰트 추가
fm.fontManager.addfont(font_path)

# 폰트 속성 설정
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["axes.unicode_minus"] = False

# Seaborn 스타일
sns.set_style('whitegrid')

print("✅ 기본 라이브러리 로드 완료!")

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

In [ ]:
fe = fm.FontEntry(fname=r'/usr/share/fonts/truetype/nanum/NanumGothic.ttf', name='NanumGothic')
fm.fontManager.ttflist.insert(0, fe)
plt.rcParams.update({'font.size': 10, 'font.family': 'NanumGothic'})

In [ ]:
# GPU 확인
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU 없음 - CPU 모드 (학습이 매우 느릴 수 있음)")

In [ ]:
# 라이브러리 설치 (약 30초 소요)
%pip install -q dagshub 'mlflow>=2,<3'

print("✅ 라이브러리 설치 완료!")

🔥 이 부분은 수정이 필요합니다.

아래 `repo_owner`, `repo_name`을 본인의 Dagshub 정보로 채워주세요.


In [ ]:
# Dagshub & MLflow 재연동
import dagshub
import mlflow

repo_owner = # 🔥 직접 작성이 필요합니다.
repo_name = # 🔥 직접 작성이 필요합니다.

dagshub.init(
    repo_owner=repo_owner,
    repo_name=repo_name,
    mlflow=True
)

mlflow.set_experiment("day1-news-classification")

print("✅ Dagshub 연동 완료!")



## 📂 1. 데이터 준비


### 데이터 불러오기

#### Google Drive

In [ ]:
# from google.colab import drive

# # Google Drive 마운트
# drive.mount('/content/drive')

# print("\n✅ Google Drive 연결 완료!")
# print("📁 Drive 경로: /content/drive/MyDrive")

In [ ]:
# import shutil
# import os

# # Google Drive에서 복사 (Drive에 업로드했을 경우)
# # drive_data_path = '/content/drive/MyDrive/dacon_data'  # Drive에 업로드한 경로
# drive_data_path = '/content/drive/MyDrive/lectures/dl_bootcamp/day1_news_classification/data/dacon_data'  # Drive에 업로드한 경로

# # 현재 작업 폴더로 복사
# if os.path.exists(f'{drive_data_path}/train_data.csv'):
#     shutil.copy(f'{drive_data_path}/train_data.csv', './train_data.csv')
#     shutil.copy(f'{drive_data_path}/test_data.csv', './test_data.csv')
#     shutil.copy(f'{drive_data_path}/topic_dict.csv', './topic_dict.csv')
#     shutil.copy(f'{drive_data_path}/sample_submission.csv', './sample_submission.csv')
#     print("✅ Drive에서 데이터 복사 완료!")
# else:
#     print("❌ Drive에 데이터가 없습니다. 방법 1을 사용하세요.")

#### 데이터 업로드

In [ ]:
import os
import zipfile
from google.colab import files

# 1. 경로 설정
# 최종적으로 데이터가 저장될 하위 폴더 이름은 'dacon_data'로 고정합니다.
base_data_path = '/content/data'
target_path = os.path.join(base_data_path, 'dacon_data')

os.makedirs(target_path, exist_ok=True)

# 2. 파일 업로드
print("📤 업로드할 ZIP 파일을 선택해주세요 (파일명 상관없음)...")
uploaded = files.upload()

# 3. 업로드된 파일 중 ZIP 파일 자동 탐색 및 압축 해제
uploaded_zip_files = [f for f in uploaded.keys() if f.endswith('.zip')]

if uploaded_zip_files:
    # 가장 먼저 업로드된 ZIP 파일 하나를 대상으로 수행
    zip_file_name = uploaded_zip_files[0]

    print(f"\n📦 '{zip_file_name}'을(를) {target_path}에 압축 해제 중...")
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall(target_path)

    print(f"✅ 압축 해제 완료: {target_path}")

    # 세션 용량 확보를 위해 업로드된 원본 zip 파일 삭제
    os.remove(zip_file_name)
else:
    print("\n⚠️ 업로드된 파일 중 ZIP 형식이 없습니다.")

# 4. 결과 확인
print(f"\n📂 {target_path} 내부 파일 목록:")
try:
    print(os.listdir(target_path))
except FileNotFoundError:
    print("폴더가 생성되지 않았습니다.")

In [ ]:
data_path = target_path

#### 데이터 불러오기

### 데이터 분리

In [ ]:
# 데이터 로드
train_df = pd.read_csv(os.path.join(data_path, 'train_data.csv'))

# Train / Validation 분리 (Day 1-2와 동일하게)
X = train_df['title']
y = train_df['topic_idx']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,  # 베이스라인과 동일한 분리를 위해 같은 seed
    stratify=y
)

print(f"✅ Train: {len(X_train):,}개")
print(f"✅ Validation: {len(X_val):,}개")

### Hugging Face Dataset 형식으로 변환

In [ ]:
from datasets import Dataset

# Pandas DataFrame → Hugging Face Dataset
train_data = pd.DataFrame({'title': X_train.values, 'label': y_train.values})
val_data = pd.DataFrame({'title': X_val.values, 'label': y_val.values})

train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

print("✅ Dataset 변환 완료!")
print(f"📊 Train Dataset: {train_dataset}")
print(f"📊 Val Dataset: {val_dataset}")


## 🤖 2. BERT 모델 및 Tokenizer 로드



### 2.1 Tokenizer 로드

https://huggingface.co/klue/bert-base

🔥 이 부분을 같이 작성해봅시다.

**model_name**과 **tokenizer** 로드를 실습 노트북에서 채워보세요. (예: `klue/bert-base`)


In [ ]:
from transformers import AutoTokenizer

model_name = # 🔥 직접 작성이 필요합니다.
tokenizer = # 🔥 직접 작성이 필요합니다.

print(f"✅ Tokenizer 로드 완료: {model_name}")
print(f"📚 Vocabulary 크기: {tokenizer.vocab_size:,}")


### Tokenizer 테스트



💡 BERT가 텍스트를 어떻게 처리하는지 확인해봅시다.

In [ ]:
# 샘플 텍스트로 테스트
sample_text = "코스피 지수 상승세 지속"

# 토큰화
tokens = tokenizer.tokenize(sample_text)
print(f"📝 원본 텍스트: {sample_text}")
print(f"🔤 토큰: {tokens}")

# ID로 변환
input_ids = tokenizer.encode(sample_text)
print(f"🔢 Token IDs: {input_ids}")

# 특수 토큰 확인
print(f"\n🔖 특수 토큰:")
print(f"  [CLS]: {tokenizer.cls_token} (ID: {tokenizer.cls_token_id})")
print(f"  [SEP]: {tokenizer.sep_token} (ID: {tokenizer.sep_token_id})")
print(f"  [PAD]: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

### 2.2 데이터 전처리 함수 정의

In [ ]:
def tokenize_function(examples):
    """
    Dataset의 title을 tokenize하는 함수
    """
    return tokenizer(
        examples['title'],
        padding='max_length',  # 최대 길이까지 패딩
        truncation=True,  # 긴 문장은 잘라냄
        max_length=128  # 뉴스 헤드라인은 128이면 충분
    )

# 전체 데이터셋에 적용 (batched=True로 빠르게 처리)
print("🔄 Tokenization 시작... (약 30초 소요)")

train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)

print("✅ Tokenization 완료!")

In [ ]:
# Tokenization 결과 확인
print("📊 Tokenized Dataset 샘플:")
print(train_tokenized[0])

# input_ids 길이 확인
print(f"\n📏 Input IDs 길이: {len(train_tokenized[0]['input_ids'])}")

🔥 이 부분을 같이 작성해봅시다.

**AutoModelForSequenceClassification.from_pretrained**로 **model**을 로드해 보세요. (num_labels=7)


In [ ]:
from transformers import AutoModelForSequenceClassification

model = # 🔥 직접 작성이 필요합니다.

model = model.to(device)

print(f"✅ 모델 로드 완료: {model_name}")
print(f"📊 파라미터 수: {model.num_parameters():,}")
print(f"🖥️ Device: {device}")



## 🏋️ 3. BERT Fine-tuning



### 3.1 학습 설정

🔥 이 부분을 같이 작성해봅시다.

**TrainingArguments**로 **training_args**를 설정해 보세요. (output_dir, num_train_epochs=3, learning_rate=2e-5 등)


In [ ]:
from transformers import TrainingArguments, Trainer

# 학습 하이퍼파라미터 설정
training_args = TrainingArguments(
    output_dir='./results',  # 체크포인트 저장 경로
    num_train_epochs=0,  # 🔥 직접 작성이 필요합니다. (예: 3)
    per_device_train_batch_size=0,  # 🔥 직접 작성이 필요합니다. (예: 16)
    per_device_eval_batch_size=0,  # 🔥 직접 작성이 필요합니다. (예: 32)
    learning_rate=0.0,  # 🔥 직접 작성이 필요합니다. (예: 2e-5)
    warmup_steps=0,  # 🔥 직접 작성이 필요합니다. (예: 500)
    weight_decay=0.0,  # 🔥 직접 작성이 필요합니다. (예: 0.01)
    logging_dir='./logs',  # 로그 저장
    logging_steps=100,  # 100 스텝마다 로그
    eval_strategy="epoch",  # 매 에포크마다 평가
    save_strategy="epoch",  # 매 에포크마다 저장
    load_best_model_at_end=True,  # 최고 성능 모델 로드
    metric_for_best_model="f1",  # F1 기준으로 최고 모델 선정
    greater_is_better=True,  # F1은 클수록 좋음
    save_total_limit=2,  # 최근 2개 체크포인트만 유지
    fp16=True if torch.cuda.is_available() else False,  # Mixed Precision (속도 향상)
    report_to="none"  # MLflow는 따로 기록할 예정
)

print("✅ 학습 설정 완료!")
print(f"📊 Epochs: {training_args.num_train_epochs}")
print(f"📊 Batch Size: {training_args.per_device_train_batch_size}")
print(f"📊 Learning Rate: {training_args.learning_rate}")

### 3.2 평가 함수 정의

🔥 이 부분을 같이 작성해봅시다.

**accuracy**, **f1_macro**를 계산하고 `{'accuracy': acc, 'f1': f1_macro}` 형태로 반환해 보세요.


In [ ]:
def compute_metrics(eval_pred):
    """
    Trainer가 사용할 평가 함수
    """
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)  # 가장 높은 확률의 클래스 선택

    acc = ...  # 🔥 직접 작성이 필요합니다. (accuracy_score)
    f1_macro = ...  # 🔥 직접 작성이 필요합니다. (f1_score, average='macro')

    return {  # 🔥 직접 작성이 필요합니다. ('accuracy', 'f1' 키로 반환)
        'accuracy': acc,
        'f1': f1_macro
    }

print("✅ 평가 함수 정의 완료!")

🔥 이 부분을 같이 작성해봅시다.

**Trainer**를 생성해 보세요. (model, args, train_dataset, eval_dataset, compute_metrics)


In [ ]:
trainer = # 🔥 직접 작성이 필요합니다.

print("✅ Trainer 생성 완료!")
print(f"📊 Train samples: {len(train_tokenized):,}")
print(f"📊 Eval samples: {len(val_tokenized):,}")


### 3.4 학습 시작! 🚀



⏱️ **예상 소요 시간**: GPU에 따라 5~15분

In [ ]:
print("🏋️ BERT Fine-tuning 시작...\n")

# 학습!
train_result = trainer.train()

print("\n✅ 학습 완료!")
print(f"📊 최종 Train Loss: {train_result.training_loss:.4f}")


## 📊 4. 모델 평가



### 4.1 Validation 성능 평가

🔥 이 부분을 같이 작성해봅시다.

**trainer.evaluate()**로 Validation 성능을 구해 **eval_result**에 담아 보세요.


In [ ]:
# Validation 평가
eval_result = # 🔥 직접 작성이 필요합니다.

print("📊 Validation 성능:")
print(f"  Accuracy: {eval_result['eval_accuracy']:.4f}")
print(f"  F1 (Macro): {eval_result['eval_f1']:.4f}")
print(f"  Loss: {eval_result['eval_loss']:.4f}")

# 목표 달성 여부
if eval_result['eval_f1'] >= 0.85:
    print("\n🎉 목표 달성! (F1 >= 0.85)")
else:
    print(f"\n⚠️ 목표 미달성 (현재: {eval_result['eval_f1']:.4f}, 목표: 0.85)")

### 4.2 예측 및 상세 평가

🔥 이 부분을 같이 작성해봅시다.

**trainer.predict(val_tokenized)**로 예측하고 **y_pred**, **y_true**를 준비해 보세요.


In [ ]:
# Validation 데이터로 예측
predictions = # 🔥 직접 작성이 필요합니다.
y_pred = # 🔥 직접 작성이 필요합니다.
y_true = # 🔥 직접 작성이 필요합니다.

# 토픽 이름
topic_names = {
    0: '정치', 1: '경제', 2: '사회', 3: '생활/문화',
    4: '세계', 5: 'IT/과학', 6: '스포츠'
}

# Classification Report
print("📋 Classification Report (BERT):")
print(classification_report(y_true, y_pred, target_names=[topic_names[i] for i in range(7)]))

### 4.3 Confusion Matrix

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=[topic_names[i] for i in range(7)],
            yticklabels=[topic_names[i] for i in range(7)])
plt.title('Confusion 행렬 - BERT 모델', fontsize=14, fontweight='bold')
plt.ylabel('실제 레이블', fontsize=12)
plt.xlabel('예측 레이블', fontsize=12)
plt.tight_layout()

# 저장
plt.savefig('confusion_matrix_bert.png', dpi=100, bbox_inches='tight')
plt.show()

# ❓ 질문: 베이스라인 대비 어떤 토픽의 성능이 가장 많이 향상되었나요?


## 🔬 5. MLflow 실험 로깅


🔥 이 부분은 수정이 필요합니다.

**run_name**을 비워두었습니다. BERT 실험을 구분하기 쉬운 이름으로 채운 뒤 Dagshub UI에서 확인해보세요.



### 5.1 BERT 실험 기록

In [ ]:
# MLflow에 BERT 실험 기록
with mlflow.start_run(run_name=""):  # 🔥 직접 작성이 필요합니다.

    # ===== Parameters =====
    mlflow.log_param('model_type', 'BERT Fine-tuning')
    mlflow.log_param('model_name', model_name)
    mlflow.log_param('num_epochs', training_args.num_train_epochs)
    mlflow.log_param('batch_size', training_args.per_device_train_batch_size)
    mlflow.log_param('learning_rate', training_args.learning_rate)
    mlflow.log_param('max_length', 128)
    mlflow.log_param('warmup_steps', training_args.warmup_steps)
    mlflow.log_param('weight_decay', training_args.weight_decay)

    # ===== Metrics =====
    mlflow.log_metric('val_accuracy', eval_result['eval_accuracy'])
    mlflow.log_metric('val_f1_macro', eval_result['eval_f1'])
    mlflow.log_metric('val_loss', eval_result['eval_loss'])
    mlflow.log_metric('train_loss', train_result.training_loss)

    # ===== Artifacts =====
    mlflow.log_artifact('confusion_matrix_bert.png')

    # Classification report
    with open('classification_report_bert.txt', 'w') as f:
        f.write(classification_report(y_true, y_pred,
                                     target_names=[topic_names[i] for i in range(7)]))
    mlflow.log_artifact('classification_report_bert.txt')

    print("✅ MLflow 실험 기록 완료!")
    print(f"🔍 Dagshub: https://dagshub.com/YOUR_USERNAME/YOUR_REPO_NAME/experiments")

### 5.2 베이스라인 vs BERT 비교



**Dagshub UI에서 비교하기**:
1. Experiments 페이지 이동
2. `baseline-tfidf-...`와 `bert-klue-...` 체크
3. "Compare" 클릭
4. F1 Score 차이 확인



**예상 결과**:

In [ ]:
# 성능 비교 요약 (수동 입력)
print("📊 성능 비교 요약:\n")
print("모델              | Val F1 | Val Accuracy")
print("-" * 45)
print(f"Baseline (TF-IDF) | ~0.80  | ~0.82")
print(f"BERT (klue)       | ~{eval_result['eval_f1']:.2f}  | ~{eval_result['eval_accuracy']:.2f}")
print("-" * 45)
print(f"향상률            | +{(eval_result['eval_f1'] - 0.80) * 100:.1f}%  | +{(eval_result['eval_accuracy'] - 0.82) * 100:.1f}%")

print("\n💡 BERT의 장점:")
print("  - 문맥 이해 (단어 순서, 의미 관계)")
print("  - 사전학습된 언어 지식 활용")
print("  - Subword 토큰화 (미등록 단어 처리)")


## 💡 6. 성능 개선 실험 (자유 시간)



### 실험 1: Learning Rate 조정

In [ ]:
# 🎯 도전 과제: learning_rate를 바꿔서 재학습
# 힌트: 3e-5, 5e-5 등으로 변경

# TODO: TrainingArguments의 learning_rate 변경 후 재학습
# 주의: 모델도 다시 로드해야 함! (이전 학습 영향 제거)

# 예시:
# model = AutoModelForSequenceClassification.from_pretrained(
#     model_name, num_labels=7
# ).to(device)
#
# training_args_new = TrainingArguments(
#     ...
#     learning_rate=3e-5,  # 변경!
#     ...
# )

In [ ]:
# 실험 1-1: Learning Rate = 3e-5
print("🔄 실험 1-1 시작: learning_rate = 3e-5\n")

# 모델 재로드 (이전 학습 영향 제거)
model_exp1 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7
).to(device)

# 학습 설정
training_args_exp1 = TrainingArguments(
    output_dir='./results_exp1',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,  # 2e-5 → 3e-5로 변경
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_exp1',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True if torch.cuda.is_available() else False,
    report_to="none"
)

# Trainer 생성
trainer_exp1 = Trainer(
    model=model_exp1,
    args=training_args_exp1,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics
)

# 학습
print("🏋️ 학습 시작...\n")
train_result_exp1 = trainer_exp1.train()

# 평가
eval_result_exp1 = trainer_exp1.evaluate()

print(f"\n✅ 실험 1-1 완료!")
print(f"📊 Val F1: {eval_result_exp1['eval_f1']:.4f}")
print(f"📊 Val Accuracy: {eval_result_exp1['eval_accuracy']:.4f}")

# MLflow 로깅
with mlflow.start_run(run_name="exp1-bert-lr3e5-ep3"):
    mlflow.log_param('model_type', 'BERT Fine-tuning')
    mlflow.log_param('model_name', model_name)
    mlflow.log_param('learning_rate', 3e-5)
    mlflow.log_param('num_epochs', 3)
    mlflow.log_param('batch_size', 16)

    mlflow.log_metric('val_f1_macro', eval_result_exp1['eval_f1'])
    mlflow.log_metric('val_accuracy', eval_result_exp1['eval_accuracy'])
    mlflow.log_metric('val_loss', eval_result_exp1['eval_loss'])

print("✅ MLflow 로깅 완료!")

In [ ]:
# 실험 1-2: Learning Rate = 5e-5
print("🔄 실험 1-2 시작: learning_rate = 5e-5\n")

# 모델 재로드
model_exp2 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7
).to(device)

# 학습 설정
training_args_exp2 = TrainingArguments(
    output_dir='./results_exp2',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,  # 2e-5 → 5e-5로 변경
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_exp2',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True if torch.cuda.is_available() else False,
    report_to="none"
)

# Trainer 생성
trainer_exp2 = Trainer(
    model=model_exp2,
    args=training_args_exp2,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics
)

# 학습
print("🏋️ 학습 시작...\n")
train_result_exp2 = trainer_exp2.train()

# 평가
eval_result_exp2 = trainer_exp2.evaluate()

print(f"\n✅ 실험 1-2 완료!")
print(f"📊 Val F1: {eval_result_exp2['eval_f1']:.4f}")
print(f"📊 Val Accuracy: {eval_result_exp2['eval_accuracy']:.4f}")

# MLflow 로깅
with mlflow.start_run(run_name="exp1-bert-lr5e5-ep3"):
    mlflow.log_param('model_type', 'BERT Fine-tuning')
    mlflow.log_param('model_name', model_name)
    mlflow.log_param('learning_rate', 5e-5)
    mlflow.log_param('num_epochs', 3)
    mlflow.log_param('batch_size', 16)

    mlflow.log_metric('val_f1_macro', eval_result_exp2['eval_f1'])
    mlflow.log_metric('val_accuracy', eval_result_exp2['eval_accuracy'])
    mlflow.log_metric('val_loss', eval_result_exp2['eval_loss'])

print("✅ MLflow 로깅 완료!")

### 실험 2: Epochs 조정

In [ ]:
# 🎯 도전 과제: num_train_epochs를 5로 늘려보기
# 주의: Overfitting 가능성 있음! Validation loss 모니터링 필수

# TODO: 여러분이 직접 작성

In [ ]:
# 실험 2-1: Epochs = 5
print("🔄 실험 2-1 시작: num_train_epochs = 5\n")

# 모델 재로드
model_exp3 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7
).to(device)

# 학습 설정
training_args_exp3 = TrainingArguments(
    output_dir='./results_exp3',
    num_train_epochs=5,  # 3 → 5로 증가
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_exp3',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True if torch.cuda.is_available() else False,
    report_to="none"
)

# Trainer 생성
trainer_exp3 = Trainer(
    model=model_exp3,
    args=training_args_exp3,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics
)

# 학습
print("🏋️ 학습 시작 (5 epochs, 시간 소요 예상)...\n")
train_result_exp3 = trainer_exp3.train()

# 평가
eval_result_exp3 = trainer_exp3.evaluate()

print(f"\n✅ 실험 2-1 완료!")
print(f"📊 Val F1: {eval_result_exp3['eval_f1']:.4f}")
print(f"📊 Val Accuracy: {eval_result_exp3['eval_accuracy']:.4f}")

# MLflow 로깅
with mlflow.start_run(run_name="exp2-bert-lr2e5-ep5"):
    mlflow.log_param('model_type', 'BERT Fine-tuning')
    mlflow.log_param('model_name', model_name)
    mlflow.log_param('learning_rate', 2e-5)
    mlflow.log_param('num_epochs', 5)
    mlflow.log_param('batch_size', 16)

    mlflow.log_metric('val_f1_macro', eval_result_exp3['eval_f1'])
    mlflow.log_metric('val_accuracy', eval_result_exp3['eval_accuracy'])
    mlflow.log_metric('val_loss', eval_result_exp3['eval_loss'])

print("✅ MLflow 로깅 완료!")

In [ ]:
# 실험 2-2: Epochs = 10 (overfitting 확인용)
print("🔄 실험 2-2 시작: num_train_epochs = 10\n")
print("⚠️ 주의: Overfitting 가능성 있음. Validation loss 모니터링 필요\n")

# 모델 재로드
model_exp4 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7
).to(device)

# 학습 설정
training_args_exp4 = TrainingArguments(
    output_dir='./results_exp4',
    num_train_epochs=10,  # 3 → 10으로 증가
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_exp4',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True if torch.cuda.is_available() else False,
    report_to="none"
)

# Trainer 생성
trainer_exp4 = Trainer(
    model=model_exp4,
    args=training_args_exp4,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics
)

# 학습
print("🏋️ 학습 시작 (10 epochs, 시간 많이 소요)...\n")
train_result_exp4 = trainer_exp4.train()

# 평가
eval_result_exp4 = trainer_exp4.evaluate()

print(f"\n✅ 실험 2-2 완료!")
print(f"📊 Val F1: {eval_result_exp4['eval_f1']:.4f}")
print(f"📊 Val Accuracy: {eval_result_exp4['eval_accuracy']:.4f}")

# MLflow 로깅
with mlflow.start_run(run_name="exp2-bert-lr2e5-ep10"):
    mlflow.log_param('model_type', 'BERT Fine-tuning')
    mlflow.log_param('model_name', model_name)
    mlflow.log_param('learning_rate', 2e-5)
    mlflow.log_param('num_epochs', 10)
    mlflow.log_param('batch_size', 16)

    mlflow.log_metric('val_f1_macro', eval_result_exp4['eval_f1'])
    mlflow.log_metric('val_accuracy', eval_result_exp4['eval_accuracy'])
    mlflow.log_metric('val_loss', eval_result_exp4['eval_loss'])

print("✅ MLflow 로깅 완료!")

### 실험 3: 다른 한국어 BERT 모델

In [ ]:
# 🎯 도전 과제: 다른 모델 시도
# 추천 모델:
#   - "klue/roberta-base" (RoBERTa, BERT의 개선 버전)
#   - "beomi/kcbert-base" (댓글/구어체 강점)

# TODO: model_name 변경 후 전체 재실행

In [ ]:
# 실험 3-1: RoBERTa 모델
print("🔄 실험 3-1 시작: klue/roberta-base\n")

model_name_exp5 = "klue/roberta-base"

# Tokenizer & Model 로드
tokenizer_exp5 = AutoTokenizer.from_pretrained(model_name_exp5)
model_exp5 = AutoModelForSequenceClassification.from_pretrained(
    model_name_exp5,
    num_labels=7
).to(device)

# 데이터 재토큰화 (토크나이저가 달라짐)
def tokenize_exp5(examples):
    return tokenizer_exp5(
        examples['title'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

train_tokenized_exp5 = train_dataset.map(tokenize_exp5, batched=True)
val_tokenized_exp5 = val_dataset.map(tokenize_exp5, batched=True)

# 학습 설정
training_args_exp5 = TrainingArguments(
    output_dir='./results_exp5',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_exp5',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True if torch.cuda.is_available() else False,
    report_to="none"
)

# Trainer 생성
trainer_exp5 = Trainer(
    model=model_exp5,
    args=training_args_exp5,
    train_dataset=train_tokenized_exp5,
    eval_dataset=val_tokenized_exp5,
    compute_metrics=compute_metrics
)

# 학습
print("🏋️ 학습 시작...\n")
train_result_exp5 = trainer_exp5.train()

# 평가
eval_result_exp5 = trainer_exp5.evaluate()

print(f"\n✅ 실험 3-1 완료!")
print(f"📊 Val F1: {eval_result_exp5['eval_f1']:.4f}")
print(f"📊 Val Accuracy: {eval_result_exp5['eval_accuracy']:.4f}")

# MLflow 로깅
with mlflow.start_run(run_name="exp3-roberta-lr2e5-ep3"):
    mlflow.log_param('model_type', 'RoBERTa Fine-tuning')
    mlflow.log_param('model_name', model_name_exp5)
    mlflow.log_param('learning_rate', 2e-5)
    mlflow.log_param('num_epochs', 3)
    mlflow.log_param('batch_size', 16)

    mlflow.log_metric('val_f1_macro', eval_result_exp5['eval_f1'])
    mlflow.log_metric('val_accuracy', eval_result_exp5['eval_accuracy'])
    mlflow.log_metric('val_loss', eval_result_exp5['eval_loss'])

print("✅ MLflow 로깅 완료!")

In [ ]:
# 실험 3-2: KcBERT 모델 (댓글/구어체 특화)
print("🔄 실험 3-2 시작: beomi/kcbert-base\n")

model_name_exp6 = "beomi/kcbert-base"

# Tokenizer & Model 로드
tokenizer_exp6 = AutoTokenizer.from_pretrained(model_name_exp6)
model_exp6 = AutoModelForSequenceClassification.from_pretrained(
    model_name_exp6,
    num_labels=7
).to(device)

# 데이터 재토큰화
def tokenize_exp6(examples):
    return tokenizer_exp6(
        examples['title'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

train_tokenized_exp6 = train_dataset.map(tokenize_exp6, batched=True)
val_tokenized_exp6 = val_dataset.map(tokenize_exp6, batched=True)

# 학습 설정
training_args_exp6 = TrainingArguments(
    output_dir='./results_exp6',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_exp6',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True if torch.cuda.is_available() else False,
    report_to="none"
)

# Trainer 생성
trainer_exp6 = Trainer(
    model=model_exp6,
    args=training_args_exp6,
    train_dataset=train_tokenized_exp6,
    eval_dataset=val_tokenized_exp6,
    compute_metrics=compute_metrics
)

# 학습
print("🏋️ 학습 시작...\n")
train_result_exp6 = trainer_exp6.train()

# 평가
eval_result_exp6 = trainer_exp6.evaluate()

print(f"\n✅ 실험 3-2 완료!")
print(f"📊 Val F1: {eval_result_exp6['eval_f1']:.4f}")
print(f"📊 Val Accuracy: {eval_result_exp6['eval_accuracy']:.4f}")

# MLflow 로깅
with mlflow.start_run(run_name="exp3-kcbert-lr2e5-ep3"):
    mlflow.log_param('model_type', 'KcBERT Fine-tuning')
    mlflow.log_param('model_name', model_name_exp6)
    mlflow.log_param('learning_rate', 2e-5)
    mlflow.log_param('num_epochs', 3)
    mlflow.log_param('batch_size', 16)

    mlflow.log_metric('val_f1_macro', eval_result_exp6['eval_f1'])
    mlflow.log_metric('val_accuracy', eval_result_exp6['eval_accuracy'])
    mlflow.log_metric('val_loss', eval_result_exp6['eval_loss'])

print("✅ MLflow 로깅 완료!")

### 베스트 모델 선정

In [ ]:
# 모든 실험 결과 비교
experiments = {
    'Baseline (BERT, lr=2e-5, ep=3)': eval_result['eval_f1'],
    'Exp 1-1 (BERT, lr=3e-5, ep=3)': eval_result_exp1['eval_f1'],
    'Exp 1-2 (BERT, lr=5e-5, ep=3)': eval_result_exp2['eval_f1'],
    'Exp 2-1 (BERT, lr=2e-5, ep=5)': eval_result_exp3['eval_f1'],
    'Exp 2-2 (BERT, lr=2e-5, ep=10)': eval_result_exp4['eval_f1'],
    'Exp 3-1 (RoBERTa, lr=2e-5, ep=3)': eval_result_exp5['eval_f1'],
    'Exp 3-2 (KcBERT, lr=2e-5, ep=3)': eval_result_exp6['eval_f1'],
}

print("📊 모든 BERT 실험 결과 요약:\n")
print(f"{'실험명':<45} {'F1 Score':>10}")
print("=" * 60)

for name, score in sorted(experiments.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:<45} {score:>10.4f}")

best_exp = max(experiments, key=experiments.get)
best_score = experiments[best_exp]

print("\n" + "=" * 60)
print(f"🏆 베스트 모델: {best_exp}")
print(f"🎯 Best F1 Score: {best_score:.4f}")
print("=" * 60)

# 베이스라인(TF-IDF)과의 비교
if 'val_f1' in dir():
    print(f"\n📈 성능 향상:")
    print(f"  TF-IDF Baseline: {val_f1:.4f}")
    print(f"  Best BERT Model: {best_score:.4f}")
    print(f"  향상률: +{(best_score - val_f1) * 100:.2f}%")


## 🎯 7. 최종 정리



### Day 1 전체 요약

In [ ]:
print("="*50)
print("🎉 Day 1 완료! 축하합니다!")
print("="*50)

print("\n✅ 오늘 배운 것:")
print("  1. MLflow 실험 관리 시스템 구축")
print("  2. 뉴스 토픽 분류 문제 이해 & EDA")
print("  3. TF-IDF 베이스라인 (F1 ~0.80)")
print("  4. BERT Fine-tuning (F1 ~0.90)")
print("  5. 체계적인 실험 비교 및 분석")

print("\n📊 최종 성적표:")
print(f"  Best Validation F1: {eval_result['eval_f1']:.4f}")
print(f"  Baseline 대비 향상: +{(eval_result['eval_f1'] - 0.80) * 100:.1f}%")

print("\n💡 핵심 인사이트:")
print("  - Transfer Learning의 위력!")
print("  - MLflow로 체계적 실험 관리 가능")
print("  - 베이스라인부터 점진적 개선")

print("\n🚀 다음 단계 (Day 2):")
print("  - NLP 생성 모델 (Seq2Seq, Transformer)")
print("  - 영어→독일어 기계 번역")
print("  - BLEU score 평가")
print("  - 생성 모델 특화 MLflow 로깅")

print("\n" + "="*50)
print("수고하셨습니다! 🎊")
print("="*50)


## ✅ Day 1-3 체크리스트



- [ ] Transformers 라이브러리 설치 및 로드
- [ ] GPU 사용 가능 확인
- [ ] Tokenizer 로드 및 테스트
- [ ] 데이터 토큰화 완료
- [ ] BERT 모델 로드
- [ ] Fine-tuning 학습 완료 (3 epochs)
- [ ] Validation F1 Score 0.85 이상 달성
- [ ] Confusion Matrix 시각화
- [ ] MLflow에 BERT 실험 기록
- [ ] Dagshub에서 Baseline vs BERT 비교
- [ ] (선택) 추가 실험 (LR, epochs, 다른 모델)




## 🔧 트러블슈팅



### GPU Out of Memory


```python
# Batch size 줄이기
per_device_train_batch_size=8  # 16 → 8

# 또는 Gradient Accumulation
gradient_accumulation_steps=2
```



### 학습이 너무 느려요


- GPU 사용 확인: `torch.cuda.is_available()`
- FP16 활성화: `fp16=True`
- Batch size 증가 (메모리 허용 범위 내)



### F1 Score가 낮아요


- Learning rate 낮추기 (1e-5)
- Epochs 늘리기 (5~10)
- 데이터 확인 (labels 매핑 오류)